# Do your routines repeat beyond chance?

**Question this notebook answers:** graph structure and motifs describe what
the agent does in aggregate. Temporal stability asks whether it keeps doing the
same things — whether the motifs from notebook 03 persist across time windows
or only describe one period.

## Why the shuffled baseline is here

Two time windows from the same agent tend to share vocabulary: same
machine, same tools, same person. A raw persistence score therefore cannot be
read on its own.

This notebook redistributes the same actions across the same windows twenty
times and reports the observed score next to that distribution. On the corpus
behind the companion article, one neighbourhood-persistence measure reached a
median of **1.000** — and the shuffled baseline returned **1.000** as well. The
action repertoire was too narrow to vary, so the measure carried no signal.

## Two effects to check before interpreting

**Session autocorrelation** — commands within one session resemble each other.
If a single session dominates a time window, the stability measured is that
session's internal coherence. `describe_windows()` flags any window where one
session exceeds 40%.

**Agent mixing** — if the windows cover different agents, the measure captures
a difference in identity rather than drift over time. Filter to one agent
before running this analysis.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SESSIONS_GLOB = "~/.openclaw/agents/*/sessions/*.jsonl"

# Paste the granularity settings from notebook 02 here.
GRANULARITY = {
    "url_segments": 3,
    "path_components": 2,
}

# Filter to a specific agent before running stability analysis.
# Mixing agents across windows measures identity differences, not time drift.
AGENT_FILTER = ""  # e.g. "c2c" — strongly recommended

# Number of time windows (2 = first half vs. second half).
N_WINDOWS = 2

# Number of random shuffles for the baseline.
TRIALS = 20
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import glob
import os

from agentgraph import build_hyperedges, temporal_stability, describe_windows

matched = glob.glob(os.path.expanduser(SESSIONS_GLOB))
if not matched:
    raise FileNotFoundError(
        f"No session files found for pattern: {SESSIONS_GLOB!r}\n"
        "Update SESSIONS_GLOB in the configuration cell above."
    )
print(f"{len(matched)} session file(s) found.")

In [ ]:
# Build hyperedges and apply the agent filter.
edges = build_hyperedges(SESSIONS_GLOB, **GRANULARITY)

if AGENT_FILTER:
    edges = [e for e in edges if e.agent == AGENT_FILTER]
    print(f"Filtered to agent '{AGENT_FILTER}': {len(edges)} hyperedges.")
else:
    print(f"Total hyperedges: {len(edges)}")
    print()
    print("WARNING: no AGENT_FILTER set.")
    print("If your traces contain more than one agent, stability results")
    print("may reflect agent identity differences rather than time drift.")

# Check for timestamps — required for stability analysis.
dated = [e for e in edges if e.timestamp]
print(f"Hyperedges with timestamps: {len(dated)} / {len(edges)}")

if not dated:
    raise ValueError(
        "No timestamps found. Stability analysis requires timestamps.\n"
        "Update iter_tool_calls() in agentgraph/extract.py to emit "
        "timestamps from your trace format."
    )

In [ ]:
# ── Step 1: window composition ───────────────────────────────────────────────
# Read the 'warning' column before interpreting the stability scores below.

windows = describe_windows(edges, n_windows=N_WINDOWS)

print(f"{'Window':>7} {'Hyperedges':>12} {'From':>12} {'To':>12} "
      f"{'Sessions':>9} {'Dom.session%':>14}  Warning")
print("-" * 85)
for w in windows:
    warning_flag = "⚠" if w["warning"] else ""
    print(
        f"{w['window']:>7} "
        f"{w['hyperedges']:>12} "
        f"{w['from']:>12} "
        f"{w['to']:>12} "
        f"{w['sessions']:>9} "
        f"{w['dominant_session_pct']:>14.1f}  "
        f"{warning_flag} {w['warning']}"
    )
    if w["agents"]:
        print(f"        agents: {w['agents']}")
    print()

# Flag the session-autocorrelation trap explicitly.
flagged = [w for w in windows if w["warning"]]
if flagged:
    print("TRAP — Session autocorrelation detected.")
    print("One session dominates a window. The 'stability' below may reflect")
    print("that session's internal coherence, not persistence over time.")
    print("Collect more sessions before interpreting the stability scores.")

In [ ]:
# ── Step 2: compute temporal stability with random baseline ──────────────────
print(f"Running {TRIALS} baseline trials... (this may take a few seconds)")

try:
    results = temporal_stability(edges, n_windows=N_WINDOWS, trials=TRIALS)
except ValueError as exc:
    print(f"\nERROR: {exc}")
    print()
    print("Solutions:")
    print("  • Collect more traces — 30 dated hyperedges per window minimum.")
    print("  • Remove AGENT_FILTER to include all agents and increase corpus size.")
    results = []

if results:
    print()
    print(f"{'Measure':<28} {'Real':>8} {'Chance mean':>12} {'±':>4} {'Chance std':>10}  Result")
    print("-" * 80)
    for r in results:
        print(r)
    print()
    above = [r for r in results if r.beats_chance]
    if above:
        print(f"{len(above)}/{len(results)} measure(s) beat the random baseline.")
    else:
        print("No measure beats the random baseline.")
        print("The agent's graph structure does not persist beyond chance.")

In [ ]:
# Optional: repeat with 3 windows for a finer temporal view.
# Requires a larger corpus (~90 dated hyperedges minimum).

if len([e for e in edges if e.timestamp]) >= 90:
    print("=" * 60)
    print("3-window analysis (early / mid / late)")
    print("=" * 60)
    try:
        results_3 = temporal_stability(edges, n_windows=3, trials=TRIALS)
        for r in results_3:
            print(r)
    except ValueError as exc:
        print(f"Skipped: {exc}")
else:
    print("Corpus too small for 3-window analysis — skipped.")

## How to read the results

Three persistence measures are computed between the first and last window:

| Measure | What it asks |
|---|---|
| **vocabulary** | Does the agent use the same atoms in both periods? (Jaccard) |
| **structure** | Do shared atoms have the same neighbours in both periods? (median Jaccard over shared nodes) |
| **weights** | Do co-occurring pairs keep the same relative frequency? (Spearman rank correlation) |

Each measure is compared to a shuffled baseline: the same hyperedges,
redistributed into the same window sizes twenty times. A score counts only if
it exceeds `chance mean + 2 × chance std`.

### The two effects, in the output

**Session autocorrelation** — if `dominant_session_pct` exceeds 40% for any
window, that window samples one task rather than a period of time. Collect more
sessions before concluding anything about temporal stability.

**Agent mixing** — if the `agents` column shows several agents in the same
windows, the score measures how far apart those agents are from each other.
Set `AGENT_FILTER` before interpreting stability.

### What the outcomes mean

- **All measures beat chance** → the graph structure persists across windows.
  The motifs from notebook 03 are candidates for procedural memory.
- **Only vocabulary beats chance** → the agent reuses the same tools and
  recombines them differently. The graph exists; its structure does not
  persist.
- **Nothing beats chance** → the corpus is too small, or the agent does not
  repeat its workflows at this granularity.